## 2) Notebook for Section 2 of Paper: Modelled Flooding

##### Before running this notebook, you need to run the following snakemake commands to produce the necessary outputs
1) snakemake -c1 flood_model_metrics_ADM0_all_countries (runs model CI analysis at ADM0 for all countries and models)
2) snakemake -c1 flood_model_admin_CI_decomposed (runs CI ADM1 decomposition for all countries and models)

##### 1) Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib as mpl
import pandas as pd
import rasterio
import geopandas as gpd
import pycountry
import numpy as np
import yaml
import os
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import matplotlib.patches as mpatches
import matplotlib as mpl
from adjustText import adjust_text

##### 2) Define Functions

In [ ]:
# Define function combining country inequality metric geopackages
def combine_national_inequality_metrics(country_list, model):
    '''
    Function for combining national metrics into one geodataframe.
    Function takes as input:
        - the list of countries to combine (ISO3 codes)
        - the flood model used (giri, jrc, or wri)
        - the metric of interest (AAR, protected_AAR, or RP100)
        - the vulnerability curve usedd (JRC, EXP, or BERN)
    '''
    # List to store individual GeoDataFrames
    gdfs = []
    # Loop over all countries in the country list
    for country in country_list:
        try:
            metric_path = os.path.join("..", "data", "results", "social_flood", "countries", f"{country}",
                                       "inequality_metrics", f"{country}_ADM0_metrics_{model}-flood_protected_AAR_V-JRC_S-rwi.gpkg")
            # Read the GeoPackage
            gdf = gpd.read_file(metric_path)
            # Append to list
            gdfs.append(gdf)
        except Exception as e:
            print(f"Error processing {country}: {e}")

    # Combine all GeoDataFrames
    if gdfs:
        combined_gdf = pd.concat(gdfs, ignore_index=True)
        return combined_gdf
    else:
        print("No valid data found")

# Define function combining country inequality metric geopackages
def combine_sub_national_inequality_metrics(country_list, model):
    '''
    Function for combining national metrics into one geodataframe.
    Function takes as input:
        - the list of countries to combine (ISO3 codes)
        - the flood model used (giri, jrc, or wri)
        - the metric of interest (AAR, protected_AAR, or RP100)
        - the vulnerability curve usedd (JRC, EXP, or BERN)
        - the admin level of the subnational (decomposed) inequality metrics
    '''
    # List to store individual GeoDataFrames
    gdfs = []
    # Loop over all countries in the country list
    for country in country_list:
        try:
            metric_path = os.path.join("..", "data", "results", "social_flood", "countries", f"{country}",
                                       "inequality_metrics", f"{country}_ADM1_admin-decomposed_metrics_{model}-flood_protected_AAR_V-JRC_S-rwi.gpkg")
            # Read the GeoPackage
            gdf = gpd.read_file(metric_path)
            # Append to list
            gdfs.append(gdf)
        except Exception as e:
            print(f"Error processing {country}: {e}")

    # Combine all GeoDataFrames
    if gdfs:
        combined_gdf = pd.concat(gdfs, ignore_index=True)
        return combined_gdf
    else:
        print("No valid data found")

def name_to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None      # or fallback string

##### 3) Load Data

In [ ]:
# Load countries from config file
with open(os.path.join("..", "config", "config.yaml"), "r") as file:
    config = yaml.safe_load(file)
countries = config.get("iso_codes", [])

# Data Preperation (combining data from multiple models)
jrc_metrics = combine_national_inequality_metrics(countries, 'jrc')
jrc_subnational_metrics = combine_sub_national_inequality_metrics(countries, 'jrc')
jrc_metrics = jrc_metrics[jrc_metrics['Population Coverage (%)'] > 90]
# jrc_metrics = jrc_metrics[~jrc_metrics['shapeName'].isin(countries_to_drop)]
jrc_metrics['ISO3'] = jrc_metrics['shapeName'].map(name_to_iso3)
# jrc_metrics['ISO3'] = jrc_metrics['ISO3'].fillna(jrc_metrics['shapeName'].map(manual_codes))
# valid_isos = jrc_metrics['ISO3'].dropna().unique() # get valid ISOs to filter subnational
jrc_subnational_metrics['model'] = 'JRC'
# jrc_subnational_metrics = jrc_subnational_metrics[jrc_subnational_metrics['ISO3'].isin(valid_isos)]
jrc_metrics['model'] = 'JRC'
giri_metrics = combine_national_inequality_metrics(countries, 'giri')
giri_subnational_metrics = combine_sub_national_inequality_metrics(countries, 'giri')
# giri_metrics = giri_metrics[~giri_metrics['shapeName'].isin(countries_to_drop)]
giri_metrics['ISO3'] = giri_metrics['shapeName'].map(name_to_iso3)
# giri_metrics['ISO3'] = giri_metrics['ISO3'].fillna(giri_metrics['shapeName'].map(manual_codes))
giri_metrics['model'] = 'GIRI'
giri_subnational_metrics['model'] = 'GIRI'
# giri_subnational_metrics = giri_subnational_metrics[giri_subnational_metrics['ISO3'].isin(valid_isos)]
giri_metrics = giri_metrics[giri_metrics['Population Coverage (%)'] > 90]
wri_metrics = combine_national_inequality_metrics(countries, 'wri')
wri_subnational_metrics = combine_sub_national_inequality_metrics(countries, 'wri')
wri_metrics = wri_metrics[wri_metrics['Population Coverage (%)'] > 90]
# wri_metrics = wri_metrics[~wri_metrics['shapeName'].isin(countries_to_drop)]
wri_metrics['ISO3'] = wri_metrics['shapeName'].map(name_to_iso3)
# wri_metrics['ISO3'] = wri_metrics['ISO3'].fillna(wri_metrics['shapeName'].map(manual_codes))
wri_metrics['model'] = 'WRI'
# wri_subnational_metrics = wri_subnational_metrics[wri_subnational_metrics['ISO3'].isin(valid_isos)]
wri_subnational_metrics['model'] = 'WRI'
# Columns to keep
cols = ["ISO3", "shapeName", "CI", "Total Flood Risk", "Population Coverage (%)", "Population", "model", 'geometry']
subnational_cols = ["ISO3", "shapeName", "shapeID", "Nat_Contrib", "Nat_RiskShare", "CI_region_only", "model", "Nat_CI", "geometry"]
jrc_metrics = jrc_metrics[cols]
jrc_subnational_metrics = jrc_subnational_metrics[subnational_cols]
giri_metrics = giri_metrics[cols]
giri_subnational_metrics = giri_subnational_metrics[subnational_cols]
wri_metrics = wri_metrics[cols]
wri_subnational_metrics = wri_subnational_metrics[subnational_cols]
# Combine the data
combined = pd.concat([jrc_metrics, giri_metrics, wri_metrics], ignore_index=True)
subnational_combined = pd.concat([jrc_subnational_metrics, giri_subnational_metrics, wri_subnational_metrics], ignore_index=True)
# Add trouble columns manually
manual_codes = {
    'Bosnia & Herzegovina': 'BIH',
    'Central African Rep':    'CAF',
    "Cote d'Ivoire":          'CIV',
    'Congo, Dem Rep of the':  'COD',
    'Congo, Rep of the':      'COG',
    'Gambia, The':            'GMB',
    'Macedonia':              'MKD',
    'Swaziland':              'SWZ',
    'Turkey':                 'TUR'
}
combined['ISO3'] = combined['ISO3'].fillna(combined['shapeName'].map(manual_codes))

##### 4) Prepare Data (compute averages and disagreement)

In [ ]:
# Calculate model averages
avg_df = (
        combined.groupby("shapeName", as_index=False)
        .agg({
            "CI": "mean",
            "Total Flood Risk": "mean",
            "Population Coverage (%)": "first",
            "Population": "first"
        })
        .rename(columns={"CI": "CI_avg", "Total Flood Risk": "FR_avg"})
)
sub_avg_df = (
            subnational_combined.groupby('shapeID', as_index=False)
            .agg({
                "Nat_Contrib": "mean",
                "Nat_RiskShare": "mean",
                "CI_region_only": "mean"
            })
            .rename(columns={"Nat_Contrib": "CI_contrib_avg", "Nat_RiskShare": "RiskShare_avg", "CI_region_only": "CI_sub_avg"})
)
# Compute sign agreement across models per country 
# Make wide table of CI per model
wide = (
    combined.pivot_table(index="shapeName", columns="model", values="CI", aggfunc="first")
    .rename(columns={"JRC": "CI_JRC", "GIRI": "CI_GIRI", "WRI": "CI_WRI"})
    .reset_index()
)

def sign_agreement(row):
    vals = np.array([row["CI_JRC"], row["CI_GIRI"], row["CI_WRI"]], dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan  # unknown
    all_pos = np.all(vals > 0)
    all_neg = np.all(vals < 0)
    all_zero = np.all(vals == 0)
    return bool(all_pos or all_neg or all_zero)

wide["sign_agree"] = wide.apply(sign_agreement, axis=1)

# Attach geometry and merge stats
geom = combined[["shapeName", "geometry"]].drop_duplicates("shapeName")
gdf_plot = geom.merge(avg_df, on="shapeName", how="left").merge(wide[["shapeName","sign_agree"]], on="shapeName", how="left")
gdf_plot = gpd.GeoDataFrame(gdf_plot, geometry="geometry", crs=combined.crs)
gdf_plot.head()
sub_geom = subnational_combined[["shapeID", "geometry"]].drop_duplicates("shapeID")
sub_gdf_plot = sub_geom.merge(sub_avg_df, on="shapeID", how="left")

##### 5) Figure 2a - National Model CI Graph

In [ ]:
# Prepare the data
bar_plot = combined.copy()
bar_plot = bar_plot.drop(columns=['Population', 'Population Coverage (%)'])
# Add a combined model column
avg_df = (bar_plot.groupby('shapeName', as_index=False)
         .agg({
             "CI": "mean",
             "Total Flood Risk": "mean", 
             "geometry": "first",
             "ISO3": "first"
         }))
avg_df["model"] = "average"
bar_plot = pd.concat([bar_plot, avg_df], ignore_index=True)
models = ("JRC","GIRI","WRI")

# Order by average
avg = (bar_plot[bar_plot['model']=="average"][['ISO3', 'CI']]
      .rename(columns={'CI':'avg'}))
order = (avg.dropna(subset=["ISO3"])
           .sort_values("avg")["ISO3"]
           .astype(str)
           .unique()
           .tolist())

# Min/max across individual models
spread = (bar_plot[bar_plot["model"].isin(models)]
         .pivot_table(index="ISO3", values="CI", aggfunc=[np.nanmin, np.nanmax]))
spread.columns = ["minCI", "maxCI"]; spread = spread.reset_index()

d = (avg.merge(spread, on="ISO3", how="left")
     .dropna(subset=["minCI","maxCI"]))
d["ISO3"] = pd.Categorical(d["ISO3"], categories=order, ordered=True)
d = d.sort_values("ISO3")

# PLOT THE CHART
x = np.arange(len(d))
fig, ax = plt.subplots(figsize=(15, 5))
# range lines
ax.vlines(x, d["minCI"], d["maxCI"], linewidth=2, alpha=0.5, label = "CI Range")
# average point
ax.scatter(x, d["avg"], s=35, label="CI Average", zorder=3)
# end caps
# ax.scatter(x, d["minCI"], s=20, marker="s", label="Min (JRC/GIRI/WRI)")
# ax.scatter(x, d["maxCI"], s=20, marker="^")

ax.axhline(0, ls="--", lw=1, alpha=0.7, color='k')
ax.set_xticks(x); ax.set_xticklabels(d['ISO3'], rotation=90, ha="center", va="top")
ax.set_xlim(-0.6, x.max()+0.6)
ax.set_ylabel("CI"); 
ax.legend(ncol=3, frameon=True, loc="upper left", bbox_to_anchor=(0.02,0.98))
ax.grid(axis="y", ls=":", lw=0.6, alpha=0.5)
plt.tight_layout(); plt.show()

##### 6) Figure 2b - National Average CI Map

In [ ]:
# Plot a map of the metrics
# Map parameters
figsize = (15, 10)
colormap = plt.cm.seismic.reversed()
colormap_min = -1
colormap_max = 1
metric = "CI_avg"
label = "National Concentration Index"
hatch_pattern = "/////"  # hashes for disagreement
mpl.rcParams['hatch.linewidth'] = 1
mpl.rcParams['hatch.color'] = 'gray'

# Set map boundaries
xmin = -120
xmax = 145
ymin = -60
ymax = 60

# Create a copy of original dataframe for mapping
map_df = gdf_plot.copy()
# Simplify the admin boundaries (useful for plotting)
map_df['geometry'] = map_df['geometry'].simplify(tolerance=0.1)

fig, ax = plt.subplots(1, 1, figsize=figsize)

# Create a basemap of the world
world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
try:
    world = gpd.read_file(world_url)
    world.plot(ax=ax, color='lightgray')
except Exception as e:
    print(f"Couldn't load world map from URL: {e}")
    print("Proceeding without background world map")

# Plot data for available countries
if metric in map_df.columns:
    # Create a mask for rows with non-null metric values
    valid_data = ~map_df[metric].isna()

    # Colormap normalizatoin
    norm = Normalize(vmin=colormap_min, vmax=colormap_max)

    # Plot coutnries with data using metric to control color
    map_df[valid_data].plot(column=metric,
                            ax=ax,
                            cmap=colormap,
                            norm=norm,
                            edgecolor='gray',
                            linewidth=0.2,
                            alpha=0.9)
    
    # 2) HATCH ONLY countries with NO sign agreement
    disagree = (map_df['sign_agree'] == False) & valid_data
    if disagree.any():
        map_df.loc[disagree].plot(
            ax=ax,
            facecolor="none",
            edgecolor='gray',
            linewidth=0.2,
            hatch=hatch_pattern,
            alpha=0.9,
        )

    sm = ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([])
    
    # Create the colorbar with custom positioning
    # cax = fig.add_axes([0.15, 0.2, 0.7, 0.03])  # [left, bottom, width, height]
    cax = fig.add_axes([0.33, 0.27, 0.33, 0.02])  # [left, bottom, width, height]
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
    # Create evenly spaced ticks (5 ticks including min and max)
    tick_values = np.linspace(colormap_min, colormap_max, 5)
    cbar.set_ticks(tick_values)
    # Format the tick labels to have fewer decimal places
    cbar.set_ticklabels([f"{v:.2f}" for v in tick_values])
    cbar.set_label(label, fontsize=12)
    cbar.ax.xaxis.set_label_position('top')

# Proxy patch for hatched disagreement countries
hatch_proxy = mpatches.Patch(
    facecolor="none",
    edgecolor="gray",
    hatch=hatch_pattern,
    label="Model CI sign disagreement"
)

ax.legend(
    handles=[hatch_proxy],
    loc="lower right",
    bbox_to_anchor=(0.95, 0.05),
    frameon=True,
    fontsize=11
)

# Set extent
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# # Add title explicitly to the ax object, not using plt.title
# ax.set_title(title, fontsize=15, pad=20)


# # Turn off the axis ticks
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
# Turn off the axis
# ax.set_axis_off()

plt.show()

##### 7) Figure 2c - Subnational average CI contribution map

In [ ]:
# Plot a map of subnatinal average model metrics
# Assing specific model to gdf_plot
# Map parameters
figsize = (15, 10)
colormap = plt.cm.seismic.reversed()
colormap_min = -0.5
colormap_max = 0.5
metric = "CI_contrib_avg"
label = "Sub-national Concentration Index Contribution"
proj = "EPSG:4326"
# Set map boundaries
xmin = -120
xmax = 145
ymin = -60
ymax = 60

# Create a copy of original dataframe for mapping
map_df = sub_gdf_plot.copy()
map_df = map_df.set_crs(4326)
map_df['geometry'] = map_df['geometry'].simplify(tolerance=0.1)
map_df = map_df.to_crs(proj) # reproject

fig, ax = plt.subplots(1, 1, figsize=figsize)

# Load world map
# Create a basemap of the world
world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
try:
    world = gpd.read_file(world_url).to_crs(proj)
    world.plot(ax=ax, color='lightgray')
except Exception as e:
    print(f"Couldn't load world map from URL: {e}")
    print("Proceeding without background world map")

# Plot data for available countries
if metric in map_df.columns:
    # Create a mask for rows with non-null metric values
    valid_data = ~map_df[metric].isna()

    # Colormap normalizatoin
    norm = Normalize(vmin=colormap_min, vmax=colormap_max)

    # Plot coutnries with data using metric to control color
    map_df[valid_data].plot(column=metric,
                            ax=ax,
                            cmap=colormap,
                            norm=norm,
                            edgecolor='gray',
                            linewidth=0.2,
                            alpha=0.9)
    
    # # 2) HATCH ONLY countries with NO sign agreement
    # disagree = (map_df['sign_agree'] == False) & valid_data
    # if disagree.any():
    #     map_df.loc[disagree].plot(
    #         ax=ax,
    #         facecolor="none",
    #         edgecolor='grey',
    #         linewidth=0.2,
    #         hatch=hatch_pattern,
    #         alpha=0.9,
    #     )

    sm = ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([])
    
    # Create the colorbar with custom positioning
    # cax = fig.add_axes([0.15, 0.2, 0.7, 0.03])  # [left, bottom, width, height]
    cax = fig.add_axes([0.33, 0.27, 0.33, 0.02])  # [left, bottom, width, height]
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
    # Create evenly spaced ticks (5 ticks including min and max)
    tick_values = np.linspace(colormap_min, colormap_max, 5)
    cbar.set_ticks(tick_values)
    # Format the tick labels to have fewer decimal places
    cbar.set_ticklabels([f"{v:.2f}" for v in tick_values])
    cbar.set_label(label, fontsize=12)
    cbar.ax.xaxis.set_label_position('top')


# Set extent
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# # Add title explicitly to the ax object, not using plt.title
# ax.set_title(title, fontsize=15, pad=20)

# # Turn off the axis ticks
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
# ax.set_axis_off()

plt.show()

##### 8) Supplementary Material - Individual Model Maps (National and Subnational)

In [ ]:
#### NATIONAL PLOT
# Plot a map of individual model metrics
# Assing specific model to gdf_plot (unhash chosen model to plot (wri, giri, or jrc))
gdf_plot = wri_metrics.copy()
# gdf_plot = giri_metrics.copy()
# gdf_plot = jrc_metrics.copy()
# Map parameters
figsize = (15, 10)
colormap = plt.cm.seismic.reversed()
colormap_min = -1
colormap_max = 1
metric = "CI"
label = "National Concentration Index"
proj = "EPSG:4326"
# Set map boundaries
xmin = -120
xmax = 145
ymin = -60
ymax = 60

# Create a copy of original dataframe for mapping
map_df = gdf_plot.copy()
map_df = map_df.set_crs(4326)
map_df['geometry'] = map_df['geometry'].simplify(tolerance=0.1)
map_df = map_df.to_crs(proj) # reproject

fig, ax = plt.subplots(1, 1, figsize=figsize)

# Load world map
# Create a basemap of the world
world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
try:
    world = gpd.read_file(world_url).to_crs(proj)
    world.plot(ax=ax, color='lightgray')
except Exception as e:
    print(f"Couldn't load world map from URL: {e}")
    print("Proceeding without background world map")

# Plot data for available countries
if metric in map_df.columns:
    # Create a mask for rows with non-null metric values
    valid_data = ~map_df[metric].isna()

    # Colormap normalizatoin
    norm = Normalize(vmin=colormap_min, vmax=colormap_max)

    # Plot coutnries with data using metric to control color
    map_df[valid_data].plot(column=metric,
                            ax=ax,
                            cmap=colormap,
                            norm=norm,
                            edgecolor='gray',
                            linewidth=0.2,
                            alpha=0.9)

    sm = ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([])
    
    # Create the colorbar with custom positioning
    cax = fig.add_axes([0.15, 0.2, 0.72, 0.03])  # [left, bottom, width, height]
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
    # Create evenly spaced ticks (5 ticks including min and max)
    tick_values = np.linspace(colormap_min, colormap_max, 5)
    cbar.set_ticks(tick_values)
    # Format the tick labels to have fewer decimal places
    cbar.set_ticklabels([f"{v:.2f}" for v in tick_values])
    cbar.set_label(label)

# Set extent
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# # Add border around the map extent
# rect = mpatches.Rectangle(
#     (xmin, ymin),                     # lower left corner
#     xmax - xmin, ymax - ymin,         # width, height
#     fill=False, color='black', linewidth=1,
#     transform=ax.transData, zorder=10
# )
# ax.add_patch(rect)

# # Add title explicitly to the ax object, not using plt.title
# ax.set_title(title, fontsize=15, pad=20)

# # Turn off the axis ticks
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_xlabel('')
# ax.set_ylabel('')
ax.set_axis_off()

plt.show()

In [ ]:
#### SUBNATIONAL PLOT
# Plot a map of subnatinal individual model metrics
# Assing specific model to gdf_plot (unhash chosen model to plot (wri, giri, or jrc))
gdf_plot = wri_subnational_metrics.copy()
# gdf_plot = giri_subnational_metrics.copy()
# gdf_plot = jrc_subnational_metrics.copy()

# Map parameters
figsize = (15, 10)
colormap = plt.cm.seismic.reversed()
colormap_min = -0.5
colormap_max = 0.5
metric = "Nat_Contrib"
label = "Sub-national Concentration Index Contribution"
proj = "EPSG:4326"
# Set map boundaries
xmin = -120
xmax = 145
ymin = -60
ymax = 60

# Create a copy of original dataframe for mapping
map_df = gdf_plot.copy()
map_df = map_df.set_crs(4326)
map_df['geometry'] = map_df['geometry'].simplify(tolerance=0.1)
map_df = map_df.to_crs(proj) # reproject

fig, ax = plt.subplots(1, 1, figsize=figsize)

# Load world map
# Create a basemap of the world
world_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
try:
    world = gpd.read_file(world_url).to_crs(proj)
    world.plot(ax=ax, color='lightgray')
except Exception as e:
    print(f"Couldn't load world map from URL: {e}")
    print("Proceeding without background world map")

# Plot data for available countries
if metric in map_df.columns:
    # Create a mask for rows with non-null metric values
    valid_data = ~map_df[metric].isna()

    # Colormap normalizatoin
    norm = Normalize(vmin=colormap_min, vmax=colormap_max)

    # Plot coutnries with data using metric to control color
    map_df[valid_data].plot(column=metric,
                            ax=ax,
                            cmap=colormap,
                            norm=norm,
                            edgecolor='gray',
                            linewidth=0.2,
                            alpha=0.9)

    sm = ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([])
    
    # Create the colorbar with custom positioning
    cax = fig.add_axes([0.15, 0.2, 0.72, 0.03])  # [left, bottom, width, height]
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
    # Create evenly spaced ticks (5 ticks including min and max)
    tick_values = np.linspace(colormap_min, colormap_max, 5)
    cbar.set_ticks(tick_values)
    # Format the tick labels to have fewer decimal places
    cbar.set_ticklabels([f"{v:.2f}" for v in tick_values])
    cbar.set_label(label)

# Set extent
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

# # Add title explicitly to the ax object, not using plt.title
# ax.set_title(title, fontsize=15, pad=20)

# # Turn off the axis ticks
# ax.set_xticks([])
# ax.set_yticks([])
# ax.set_xlabel('')
# ax.set_ylabel('')
ax.set_axis_off()

plt.show()